# S6-06 FinOps plane draft POC — HOBO-661

The FinOps plane closes the loop between **plan-time cost estimates** (priced from a
provider's `cost.json`), **actual spend** (a showback projection over `meter.UsageRecorded`
events, reconciled against the MeteringView), and **budget enforcement** (a deploy gate that
allows, denies, or requires approval). Everything below runs the package's real modules on
the committed `nfr-012-plan-vs-actual` scenario fixture in `test/fixtures/`.

In [1]:
import { strict as assert } from "node:assert";
import { estimatePlanCost, loadFinOpsPlaneScenario, runFinOpsPlaneScenario, withinCostTolerance } from "./src/index.ts";

const scenario = loadFinOpsPlaneScenario();
const result = runFinOpsPlaneScenario(scenario);
console.log(`scenario: ${result.scenarioId}`);
for (const [bullet, held] of Object.entries(result.acceptance)) {
  assert.equal(held, true);
  console.log(`acceptance ${bullet}: ${held}`);
}

// NFR-012: the plan estimate lands within +/-20% of recorded actual spend.
assert.equal(withinCostTolerance(result.planEstimate.projectedMonthlyUsd, scenario.expectedActualUsd, 0.2), true);
console.log(`plan vs actual: projected=$${result.planEstimate.projectedMonthlyUsd} actual=$${result.actualUsd}`);

// Fail closed: a plan region missing from cost.json coverage refuses to price.
const resources = scenario.plan.resources.map(({ region, ...resource }) => resource);
try {
  estimatePlanCost({ costJson: scenario.costJson, plan: { ...scenario.plan, regions: ["eu-west-1"], resources } });
} catch (error) {
  console.log(`uncovered region rejected: ${error.message}`);
}

scenario: nfr-012-plan-vs-actual
acceptance planEstimateWithinTolerance: true
acceptance budgetBreachBlocksDeploy: true
acceptance showbackReconciled: true
acceptance deployAllowedUnderBudget: true
plan vs actual: projected=$88.05 actual=$84
uncovered region rejected: plan region missing from cost.json coverage: eu-west-1


## Showback reconciliation and the budget deploy gate

Showback totals must equal the MeteringView projection to the cent. The deploy gate then
turns budgets into decisions: allow under budget, **deny** when existing spend plus the
estimate crosses a hard limit, and **require-approval** on an agent spend-cap soft breach.

In [2]:
import { strict as assert } from "node:assert";
import { evaluateFinOpsDeployGate, loadFinOpsPlaneScenario, projectShowbackFromEvents, runFinOpsPlaneScenario } from "./src/index.ts";

const scenario = loadFinOpsPlaneScenario();
const result = runFinOpsPlaneScenario(scenario);
const rec = result.showbackReconciliation;
assert.equal(rec.showbackTotalUsd, rec.meteringTotalUsd);
console.log(`showback reconciled=${rec.reconciled} showbackTotal=$${rec.showbackTotalUsd} meteringTotal=$${rec.meteringTotalUsd}`);
console.log(`gate under budget: effect=${result.allowGate.effect} blocksDeploy=${result.allowGate.blocksDeploy}`);

const projection = projectShowbackFromEvents({
  events: scenario.meteringEvents,
  seq: scenario.meteringEvents.length,
  costCatalog: { currency: scenario.costJson.currency, unitPrices: scenario.costJson.unitPrices },
  displayNames: scenario.displayNames,
});

const denyGate = evaluateFinOpsDeployGate({
  costEstimateUsd: 15,
  meteringView: { ...projection.meteringView, scopeCosts: { "camp:tenant-a/camp-prod": { cost: 190, currency: "USD" } } },
  policies: [{
    id: "camp-budget:tenant-a/camp-prod",
    scope: { kind: "camp", id: "tenant-a/camp-prod" },
    hardLimitUsd: 200,
    lastEvaluatedSeq: projection.meteringView.seq,
  }],
  owner: scenario.meteringEvents[0].owner,
});
console.log(`gate over hard limit: effect=${denyGate.effect} violation="${denyGate.violations[0]}"`);

const softGate = evaluateFinOpsDeployGate({
  costEstimateUsd: 5,
  meteringView: { ...projection.meteringView, scopeCosts: { "agent:tenant-a/agent-deploy": { cost: 28, currency: "USD" } } },
  policies: [{
    id: "agent-cap:tenant-a/agent-deploy",
    scope: { kind: "agent", id: "tenant-a/agent-deploy" },
    spendCapUsd: 30,
    allowedApprovers: ["user:finance-admin"],
    lastEvaluatedSeq: projection.meteringView.seq,
  }],
  owner: scenario.meteringEvents[0].owner,
});
console.log(`gate agent soft cap: effect=${softGate.effect} requiresApproval=${softGate.requiresApproval}`);
console.log("CONTRACT HELD: finops estimates within tolerance, reconciles showback, and gates deploys on budget");

showback reconciled=true showbackTotal=$84 meteringTotal=$84
gate under budget: effect=allow blocksDeploy=false
gate over hard limit: effect=deny violation="camp-budget:tenant-a/camp-prod: projected spend 205 exceeds hard budget 200"
gate agent soft cap: effect=require-approval requiresApproval=true
CONTRACT HELD: finops estimates within tolerance, reconciles showback, and gates deploys on budget
